In [1]:
# @title Imports and Notebook Utilities
# This block is mostly taken from the self-org textures notebook.
import os
import io
import PIL.Image, PIL.ImageDraw
import base64
import zipfile
import json
import requests
import numpy as np
import matplotlib.pylab as pl
import glob

from IPython.display import Image, HTML, Markdown, clear_output
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")

os.environ['FFMPEG_BINARY'] = 'ffmpeg'
import moviepy.editor as mvp
from moviepy.video.io.ffmpeg_writer import FFMPEG_VideoWriter


def imread(url, max_size=None, mode=None):
    if isinstance(url, str) and url.startswith(('http:', 'https:')):
        # wikimedia requires a user agent
        headers = {
            "User-Agent": "Requests in Colab/0.0 (https://colab.research.google.com/; no-reply@google.com) requests/0.0"
        }
        r = requests.get(url, headers=headers)
        f = io.BytesIO(r.content)
    else:
        f = url
    img = PIL.Image.open(f)
    if max_size is not None:
        img.thumbnail((max_size, max_size), PIL.Image.LANCZOS)
    if mode is not None:
        img = img.convert(mode)
    img = np.float32(img) / 255.0
    return img


def np2pil(a):
    if a.dtype in [np.float32, np.float64]:
        a = np.uint8(np.clip(a, 0, 1) * 255)
    return PIL.Image.fromarray(a)


def imwrite(f, a, fmt=None):
    a = np.asarray(a)
    if isinstance(f, str):
        fmt = f.rsplit('.', 1)[-1].lower()
        if fmt == 'jpg':
            fmt = 'jpeg'
        f = open(f, 'wb')
    np2pil(a).save(f, fmt, quality=95)


def imencode(a, fmt='jpeg'):
    a = np.asarray(a)
    if len(a.shape) == 3 and a.shape[-1] == 4:
        fmt = 'png'
    f = io.BytesIO()
    imwrite(f, a, fmt)
    return f.getvalue()


def im2url(a, fmt='jpeg'):
    encoded = imencode(a, fmt)
    base64_byte_string = base64.b64encode(encoded).decode('ascii')
    return 'data:image/' + fmt.upper() + ';base64,' + base64_byte_string


def imshow(a, fmt='jpeg', id=None):
    return display(Image(data=imencode(a, fmt)), display_id=id)


def grab_plot(close=True):
    """Return the current Matplotlib figure as an image"""
    fig = pl.gcf()
    fig.canvas.draw()
    img = np.array(fig.canvas.renderer._renderer)
    a = np.float32(img[..., 3:] / 255.0)
    img = np.uint8(255 * (1.0 - a) + img[..., :3] * a)  # alpha
    if close:
        pl.close()
    return img



def zoom(img, scale=4):
    img = np.repeat(img, scale, 0)
    img = np.repeat(img, scale, 1)
    return img


class VideoWriter:
    def __init__(self, filename='_autoplay.mp4', fps=30.0, **kw):
        self.writer = None
        self.params = dict(filename=filename, fps=fps, **kw)

    def add(self, img):
        img = np.asarray(img)
        if self.writer is None:
            h, w = img.shape[:2]
            self.writer = FFMPEG_VideoWriter(size=(w, h), **self.params)
        if img.dtype in [np.float32, np.float64]:
            img = np.uint8(img.clip(0, 1) * 255)
        if len(img.shape) == 2:
            img = np.repeat(img[..., None], 3, -1)
        self.writer.write_frame(img)

    def close(self):
        if self.writer:
            self.writer.close()

    def __enter__(self):
        return self

    def __exit__(self, *kw):
        self.close()
        if self.params['filename'] == '_autoplay.mp4':
            self.show()

    def show(self, **kw):
        self.close()
        fn = self.params['filename']
        display(mvp.ipython_display(fn, **kw))

!nvidia-smi -L

GPU 0: NVIDIA L4 (UUID: GPU-5a006382-6584-7e3e-88cf-0c14bd72b7ed)


In [2]:
import torch
import torchvision.models as models

torch.set_default_tensor_type('torch.cuda.FloatTensor')

In [ ]:
#@title VGG16 Sliced OT Style Loss
import torch.nn.functional as F

def calc_styles_vgg(imgs, vgg):
    style_layers = [1, 6, 11, 18, 25]
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    x = (imgs - mean) / std
    b, c, h, w = x.shape
    features = [x.reshape(b, c, h * w)]
    for i, layer in enumerate(vgg[:max(style_layers) + 1]):
        x = layer(x)
        if i in style_layers:
            b, c, h, w = x.shape
            features.append(x.reshape(b, c, h * w))
    return features

def project_sort(x, proj):
    return torch.einsum('bcn,cp->bpn', x, proj).sort()[0]

def ot_loss(source, target, proj_n=32):
    ch, n = source.shape[-2:]
    projs = F.normalize(torch.randn(ch, proj_n), dim=0)
    source_proj = project_sort(source, projs)
    target_proj = project_sort(target, projs)
    target_interp = F.interpolate(target_proj, n, mode='nearest')
    return (source_proj - target_interp).square().sum()

def create_vgg_loss(vgg, target_img):
    yy = calc_styles_vgg(target_img, vgg)
    def loss_f(imgs):
        xx = calc_styles_vgg(imgs, vgg)
        return sum(ot_loss(x, y) for x, y in zip(xx, yy))
    return loss_f


In [ ]:
#@title Load VGG and Target image {vertical-output: true}
vgg = models.vgg16(weights='IMAGENET1K_V1').features

from google.colab import files

print("Please upload your style image:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print(f'Using uploaded file: "{filename}"')

style_img = imread(io.BytesIO(uploaded[filename]), max_size=128)
style_img_torch = torch.tensor(style_img).permute(2, 0, 1).unsqueeze(0)

with torch.no_grad():
  loss_fn = create_vgg_loss(vgg, style_img_torch)
imshow(style_img)

In [5]:
#@title NoiseNCA Architecture
def depthwise_conv(x, filters):
    """filters: [filter_n, h, w]"""
    b, ch, h, w = x.shape
    y = x.reshape(b * ch, 1, h, w)
    y = torch.nn.functional.pad(y, [1, 1, 1, 1], "circular")
    y = torch.nn.functional.conv2d(y, filters[:, None])
    return y.reshape(b, -1, h, w)

def merge_lap(z):
    # This function merges the lap_x and lap_y into a single laplacian filter
    b, c, h, w = z.shape # [b, 5 * chn, h, w]
    z = torch.stack([
        z[:, ::5],
        z[:, 1::5],
        z[:, 2::5],
        z[:, 3::5] + z[:, 4::5]
    ], dim=2)  # [b, chn, 4, h, w]
    return z.reshape(b, -1, h, w)  # [b, 4 * chn, h, w]


class NoiseNCA(torch.nn.Module):
    def __init__(self, chn=12, fc_dim=96, noise_level=0.1):
        super().__init__()
        self.chn = chn
        self.register_buffer("noise_level", torch.tensor([noise_level]))
        self.w1 = torch.nn.Conv2d(chn * 4, fc_dim, 1, bias=True)
        self.w2 = torch.nn.Conv2d(fc_dim, chn, 1, bias=False)

        torch.nn.init.xavier_normal_(self.w1.weight, gain=0.2)
        torch.nn.init.zeros_(self.w2.weight)

        with torch.no_grad():
            ident = torch.tensor([[0.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 0.0]])
            sobel_x = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]])
            lap_x = torch.tensor([[1.0, 2.0, 1.0], [2.0, -12.0, 2.0], [1.0, 2.0, 1.0]]) # alex says that unnormalized filters work better

            self.filters = torch.stack([ident, sobel_x, sobel_x.T, lap_x, lap_x.T])

    def perception(self, s, dx=1.0, dy=1.0):
        z = depthwise_conv(s, self.filters)  # [b, 5 * chn, h, w]
        if isinstance(dx, float) and dx == 1.0 and isinstance(dy, float) == 1.0:
            return merge_lap(z)

        if not isinstance(dx, torch.Tensor) or dx.ndim != 3:
            dx = torch.tensor([dx], device=s.device)[:, None, None]  # [1, 1, 1]
        if not isinstance(dy, torch.Tensor) or dy.ndim != 3:
            dy = torch.tensor([dy], device=s.device)[:, None, None]  # [1, 1, 1]

        scale = 1.0 / torch.stack([torch.ones_like(dx), dx, dy, dx ** 2, dy ** 2], dim=1)
        scale = torch.tile(scale, (1, self.chn, 1, 1))
        z = z * scale
        return merge_lap(z)

    def forward(self, s, dx=1.0, dy=1.0, dt=1.0, noise=None):
      if noise is not None:
        s += torch.randn_like(s) * noise
      z = self.perception(s, dx, dy)
      delta_s = self.w2(torch.relu(self.w1(z)))
      return s + delta_s * dt


    def seed(self, n, h=128, w=128):
        return (torch.rand(n, self.chn, h, w) - 0.5) * self.noise_level

def to_rgb(s):
    return s[..., :3, :, :] + 0.5

param_n = sum(p.numel() for p in NoiseNCA().parameters())
print('NoiseNCA param count:', param_n)

NoiseNCA param count: 5856


In [ ]:
#@title Setup Training
import os
from google.colab import files

# Check if there are any checkpoint files in the current directory
checkpoint_files = glob.glob('*.pt')

if checkpoint_files:
    print(f"Found {len(checkpoint_files)} checkpoint file(s) in Colab storage:")
    for i, f in enumerate(checkpoint_files):
        file_size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  [{i}] {f} ({file_size_mb:.2f} MB)")
    
    choice = input("\nEnter the number to load a checkpoint, 'u' to upload a new file, or press Enter to start fresh: ").strip()
    
    if choice == 'u':
        print("Please upload your checkpoint file:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        print(f'Loading model from uploaded file: "{filename}"')
        model = torch.load(filename)
    elif choice.isdigit() and 0 <= int(choice) < len(checkpoint_files):
        filename = checkpoint_files[int(choice)]
        print(f'Loading model from: "{filename}"')
        model = torch.load(filename)
    else:
        print('Initializing new NoiseNCA model...')
        model = NoiseNCA()
else:
    print("No checkpoint files found in Colab storage.")
    upload_choice = input("Upload a checkpoint file? (y/n, default=n): ").strip().lower()
    
    if upload_choice == 'y':
        print("Please upload your checkpoint file:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        print(f'Loading model from: "{filename}"')
        model = torch.load(filename)
    else:
        print('Initializing new NoiseNCA model...')
        model = NoiseNCA()

opt = torch.optim.Adam(model.parameters(), 1e-3, capturable=True)

# ReduceLROnPlateau: adaptive learning rate scheduler
# - Monitors the loss and reduces LR when it plateaus
# - patience=500: Wait 500 steps with no improvement before reducing LR
# - factor=0.3: Multiply LR by 0.3 when reducing (same as MultiStepLR)
# - threshold=0.01: Loss must improve by >1% to count as improvement
# - min_lr=1e-6: Don't reduce LR below this value
lr_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt, 
    mode='min',           # We want to minimize the loss
    factor=0.3,           # Multiply LR by 0.3 when reducing
    patience=500,         # Wait 500 steps without improvement
    threshold=0.01,       # Must improve by >1% to count
    threshold_mode='rel', # Relative threshold (1% relative improvement)
    min_lr=1e-6,          # Don't go below this LR
    verbose=True          # Print when LR changes
)

loss_log = []
with torch.no_grad():
    pool = model.seed(256)

In [ ]:
# @title Training loop {vertical-output: true}
# This block is adapted from the self-org textures notebook.

for i in range(5000):
    with torch.no_grad():
        batch_idx = np.random.choice(len(pool), 4, replace=False)
        s = pool[batch_idx]
        if i % 8 == 0:
            s[:1] = model.seed(1)
    step_n = np.random.randint(32, 96)
    for k in range(step_n):
        s = model(s,noise=0.05) # set noise level experienced during training


    overflow_loss = (s - s.clamp(-1.0, 1.0)).abs().sum()
    loss = loss_fn(to_rgb(s)) + overflow_loss
    with torch.no_grad():
        loss.backward()
        for p in model.parameters():
            p.grad /= (p.grad.norm() + 1e-8)  # normalize gradients
        opt.step()
        opt.zero_grad()
        lr_sched.step(loss)  # Pass loss to adaptive scheduler
        pool[batch_idx] = s  # update pool

        loss_log.append(loss.item())
        if i % 5 == 0:
            # Get current LR from optimizer (scheduler doesn't have get_last_lr for ReduceLROnPlateau)
            current_lr = opt.param_groups[0]['lr']
            display(Markdown(f'''
        step_n: {len(loss_log)}
        loss: {loss.item():.2e}
        lr: {current_lr:.2e}'''), display_id='stats')
        if i % 32 == 0:
            pl.plot(loss_log, '.', alpha=0.1)
            pl.yscale('log')
            pl.ylim(np.min(loss_log), loss_log[0])
            pl.tight_layout()
            imshow(grab_plot(), id='log')
            imgs = to_rgb(s).permute([0, 2, 3, 1]).cpu()
            imshow(np.hstack(imgs), id='batch')

In [10]:
import torch
from google.colab import files

# Define the filename for the saved model weights
model_filename = 'weights.pt'

# Save the model's state_dict
torch.save(model.state_dict(), model_filename)

# Offer to download the file
print(f"Model weights saved as '{model_filename}'. You can download it below:")
files.download(model_filename)

Model weights saved as 'weights.pt'. You can download it below:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>